In [1]:
import os
os.environ["MASTER_ADDR"] = "localhost"
os.environ["MASTER_PORT"] = "29400"

In [2]:
import torch
import torch.distributed as dist
import torch.multiprocessing as mp

def setup_dist(rank, world_size):
    dist.init_process_group(
        backend='nccl', 
        # init_method='env://', 
        rank=rank, 
        world_size=world_size
    )

def cleanup_dist():
    dist.destroy_process_group()

In [3]:
from xfuser import xFuserCogVideoXPipeline, xFuserArgs

/home/cyril-k/.cache/pypoetry/virtualenvs/run-xdit-JAkJdIAX-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def run_dist(rank, world_size):
    setup_dist(rank, world_size)
    
    args = xFuserArgs(
        model="THUDM/CogVideoX-5b",
        tensor_parallel_degree=world_size,
        ulysses_degree=1,
        ring_degree=1,
        height=480,
        width=720,
        num_frames=9,
        num_inference_steps=20,
        warmup_steps=0,
        prompt="A small dog",
    )

    engine_config, input_config = args.create_config()
    cleanup_dist()

In [5]:
import cvx

if __name__ == "__main__":
    world_size = torch.cuda.device_count()
    # mp.set_start_method('fork', force=True)
    mp.spawn(
        cvx.run_dist, 
        args=(world_size,), 
        nprocs=world_size, 
        join=True,
    )
    

INFO 10-06 10:56:32 [config.py:148] Pipeline patch number not set, using default value 1
INFO 10-06 10:56:32 [config.py:148] Pipeline patch number not set, using default value 1


W1006 10:56:33.022000 140118997763200 torch/multiprocessing/spawn.py:146] Terminating process 230038 via signal SIGTERM


ProcessRaisedException: 

-- Process 1 terminated with the following error:
Traceback (most recent call last):
  File "/home/cyril-k/.cache/pypoetry/virtualenvs/run-xdit-JAkJdIAX-py3.10/lib/python3.10/site-packages/torch/multiprocessing/spawn.py", line 76, in _wrap
    fn(i, *args)
  File "/home/cyril-k/projects/run-xdit/notebooks/cvx.py", line 48, in run_dist
    local_rank = get_world_group().local_rank
  File "/home/cyril-k/.cache/pypoetry/virtualenvs/run-xdit-JAkJdIAX-py3.10/lib/python3.10/site-packages/xfuser/core/distributed/parallel_state.py", line 37, in get_world_group
    assert _WORLD is not None, "world group is not initialized"
AssertionError: world group is not initialized
